In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom automatically)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm

# ── Load dataset ──────────────────────────────────────────────────────────────
hitters = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Hitters.csv').dropna()
print(f'✓ Hitters: {hitters.shape}')

print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


In [ ]:
# ⚠️  Run this cell first before any others
# (Tip: Runtime → Run all  OR  Shift+Enter from the top)
import pandas as pd, numpy as np
hitters = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Hitters.csv').dropna()
hitters = pd.get_dummies(hitters, drop_first=True, dtype=float)
X = hitters.drop('Salary', axis=1)
y = np.log(hitters['Salary'])
print(f"Hitters: {X.shape}  |  Predicting: log(Salary)")

In [ ]:
# Show coefficient paths as lambda varies
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

alphas = np.logspace(-3, 5, 200)
ridge_coefs = []
lasso_coefs = []

for a in alphas:
    ridge_coefs.append(Ridge(alpha=a).fit(X_tr_s, y_tr).coef_)
    try:
        lasso_coefs.append(Lasso(alpha=a, max_iter=10000).fit(X_tr_s, y_tr).coef_)
    except:
        lasso_coefs.append(np.zeros(X_tr_s.shape[1]))

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(ridge_coefs.shape[1]):
    axes[0].plot(np.log10(alphas), ridge_coefs[:,i], lw=0.8, alpha=0.7)
axes[0].axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
axes[0].set_xlabel('log₁₀(λ)')
axes[0].set_ylabel('Standardized Coefficient')
axes[0].set_title('Ridge — coefficients shrink smoothly\n(none reach exactly zero)')

for i in range(lasso_coefs.shape[1]):
    axes[1].plot(np.log10(alphas), lasso_coefs[:,i], lw=0.8, alpha=0.7)
axes[1].axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
axes[1].set_xlabel('log₁₀(λ)')
axes[1].set_ylabel('Standardized Coefficient')
axes[1].set_title('Lasso — coefficients reach exactly zero\n(automatic feature selection!)')

plt.suptitle('Coefficient Paths: Ridge vs Lasso', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 Key difference: Lasso produces sparse solutions (some β=0 exactly).")
print("   Ridge keeps all predictors but shrinks them.")

In [ ]:
# Choose optimal lambda via cross-validation
ridge_cv = RidgeCV(alphas=np.logspace(-3, 5, 100), cv=10)
lasso_cv = LassoCV(alphas=np.logspace(-4, 2, 100), cv=10, max_iter=10000)

ridge_cv.fit(X_tr_s, y_tr)
lasso_cv.fit(X_tr_s, y_tr)

print(f"Ridge optimal λ: {ridge_cv.alpha_:.4f}")
print(f"Lasso optimal λ: {lasso_cv.alpha_:.4f}")
print(f"Lasso zeroed out: {np.sum(lasso_cv.coef_ == 0)}/{X.shape[1]} predictors")

# Compare test MSE
ols   = LinearRegression().fit(X_tr_s, y_tr)
models_res = {
    'OLS':   mean_squared_error(y_te, ols.predict(X_te_s)),
    'Ridge': mean_squared_error(y_te, ridge_cv.predict(X_te_s)),
    'Lasso': mean_squared_error(y_te, lasso_cv.predict(X_te_s)),
}

print("\n=== Test MSE Comparison ===")
for name, mse in models_res.items():
    print(f"  {name:<8} MSE = {mse:.4f}  {'← best' if mse == min(models_res.values()) else ''}")

In [ ]:
# Visualize: which predictors Lasso keeps vs zeros
coef_df = pd.DataFrame({
    'Predictor': X.columns,
    'OLS': ols.coef_,
    'Ridge': ridge_cv.coef_,
    'Lasso': lasso_cv.coef_
}).sort_values('Lasso', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
x_pos = np.arange(len(coef_df))
w = 0.28
ax.bar(x_pos - w, coef_df['OLS'],   w, label='OLS',   color='#888',    alpha=0.7)
ax.bar(x_pos,     coef_df['Ridge'], w, label='Ridge',  color='#1e5fa8', alpha=0.8)
ax.bar(x_pos + w, coef_df['Lasso'], w, label='Lasso',  color='#e85d20', alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(coef_df['Predictor'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Standardized Coefficient')
ax.set_title('OLS vs Ridge vs Lasso Coefficients')
ax.legend()
plt.tight_layout()
plt.show()
print("\n📌 Orange bars at zero = Lasso has eliminated that predictor entirely.")

In [ ]:
# @title 📝 Quiz — Ridge & Lasso
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What penalty does Ridge add to the OLS objective?
# @markdown **Q2:** Why does Lasso produce exactly-zero coefficients but Ridge does not?
# @markdown **Q3:** How do you choose the optimal λ in practice?
# @markdown **Q4:** When would you prefer Ridge over Lasso?
# @markdown **Q5:** What happens to all coefficients as λ → ∞?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

In [ ]:
_NB_NAME_="Ridge & Lasso"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*